In [1]:
%load_ext autoreload
%autoreload 2

In [10]:
import numpy as np
import torch
from models.linear_probes import linear_probe, linear_probe_tuned
from models.feature_generation import build_feature_bank, extract_encoder, extract_feature,pool_features
from preprocessing.dataset import PipistrelleDataset
from evaluation.metrics import compute_cv_stats,plot_model_comparison,label_confusion
import pandas as pd
from sklearn.metrics import average_precision_score
from evaluation.metrics import compute_metrics,compile_results,generate_metrics_table2,plot_comprehensive_boxplots
from evaluation.metrics import plot_comprehensive_calibration
from models.MLP_balancing import balancing_mlp
import pickle
from pathlib import Path
import os
from evaluation.statistical_tests import perform_encoder_statistical_analysis
from preprocessing.dataset import BatAudioPipeline
from models.abmil_model import train_abmil_all_data, evaluate_abmil_ood

In [3]:
import pickle

pickle_filename = 'perch2-bags.pkl'
with open(pickle_filename, 'rb') as pickle_file:
    X_bags = pickle.load(pickle_file)

In [4]:
X_bags_pooled = pool_features(X_bags, windows  = False,window_pooled  = False, method  ='mean',encoder  = 'perch2')

In [5]:
print(len(X_bags))

284


In [6]:
dir = Path(os.getcwd()).resolve().parent 
path = str(dir / "models" / "features")
y = np.load(path + "\\Y_labels2_not_normalized.npy")

In [7]:
clf = train_abmil_all_data(X_bags_pooled, y, random_state=42)

Training production model on all available data...


In [8]:
#extract ood perch 2 embeddings
device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder = "perch2"
encoder_model = extract_encoder(encoder)
dir = Path(os.getcwd()).resolve().parent / "data"
batdata = PipistrelleDataset(data_input=str(dir / "ood_metadata.csv"),
                             root_dir=str(dir / "domain-transfer-dataset"),
                             encoder = encoder,
                             filter_echo = False)

X_bags_ood,labels = build_feature_bank(batdata, encoder_model, encoder, device=device)

Dataset type: <class 'preprocessing.dataset.PipistrelleDataset'>


Extracting perch2: 100%|██████████| 673/673 [1:00:44<00:00,  5.42s/it]


In [ ]:
pickle_filename = 'perch2-ood-bags.pkl'
with open(pickle_filename, 'wb') as pickle_file:
    pickle.dump(X_bags_ood, pickle_file)


In [13]:
pickle_filename = 'perch2-ood-label.pkl'
with open(pickle_filename, 'wb') as pickle_file:
    pickle.dump(labels, pickle_file)

In [11]:
X_bags_pooled_ood = pool_features(X_bags_ood, windows=False, window_pooled=False, method='mean',encoder="perch2")

In [12]:
y_pred_proba_ood, attention_weights_ood = evaluate_abmil_ood(clf,X_bags_pooled_ood) 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import average_precision_score

def generate_perfect_multilabel_tables(y_pred_proba, metadata_csv="ood_metadata.csv"):
    df = pd.read_csv(metadata_csv)
    class_cols = ['type_a', 'type_b', 'type_c', 'type_d', 'echo']
    
    # Map predictions to columns
    for idx, col in enumerate(class_cols):
        df[f'pred_{col}'] = y_pred_proba[:, idx]
        
    specialized_keywords = [
        'B.in_flight', 'C.i', 'C.d_stationary', 
        'C.d_flying', 'C.d_advertisement', 
        'D.thrills_nathusius', 'D.part_e_nathusius'
    ]
    
    # Route multi-label combinations containing specialized tags to Table 2
    df['is_specialized'] = df['subtype'].apply(lambda x: any(kw in str(x) for kw in specialized_keywords))
    
    def isolate_specialized_name(subtype_str):
        for kw in specialized_keywords:
            if kw in str(subtype_str):
                return kw
        return subtype_str
        
    df['clean_specialized_group'] = df['subtype'].apply(isolate_specialized_name)
    
    df_normal_recordings = df[~df['is_specialized']].copy()
    df_specialized_recordings = df[df['is_specialized']].copy()
    
    # Advanced AP calculation logic using Global Negative Context Fallback
    def build_robust_ap_matrix(df_slice, full_df, group_by_col):
        rows = []
        for group_name, group_data in df_slice.groupby(group_by_col):
            if len(group_data) < 1:
                continue
                
            entry = {group_by_col: group_name}
            for col in class_cols:
                y_true_group = group_data[col].values
                y_pred_group = group_data[f'pred_{col}'].values
                
                # Condition A: Both 0 and 1 exist inside the group slice
                if len(np.unique(y_true_group)) > 1:
                    score = average_precision_score(y_true_group, y_pred_group)
                    entry[f'{col} AP'] = f"{score:.4f}"
                    
                # Condition B: The slice contains only positive examples (all 1s)
                elif np.all(y_true_group == 1):
                    # Gather all true negative samples from across the entire dataset
                    global_negs = full_df[full_df[col] == 0]
                    if not global_negs.empty:
                        y_true_eval = np.concatenate([y_true_group, global_negs[col].values])
                        y_pred_eval = np.concatenate([y_pred_group, global_negs[f'pred_{col}'].values])
                        score = average_precision_score(y_true_eval, y_pred_eval)
                        entry[f'{col} AP'] = f"{score:.4f}*"  # Asterisk flags global context usage
                    else:
                        entry[f'{col} AP'] = "-"
                        
                # Condition C: The slice contains only negative examples (all 0s)
                else:
                    # Gather all true positive samples from across the entire dataset
                    global_pos = full_df[full_df[col] == 1]
                    if not global_pos.empty:
                        y_true_eval = np.concatenate([y_true_group, global_pos[col].values])
                        y_pred_eval = np.concatenate([y_pred_group, global_pos[f'pred_{col}'].values])
                        score = average_precision_score(y_true_eval, y_pred_eval)
                        entry[f'{col} AP'] = f"{score:.4f}*"
                    else:
                        entry[f'{col} AP'] = "-"
            rows.append(entry)
        return pd.DataFrame(rows)

    # Generate both tables
    table_1 = build_robust_ap_matrix(df_normal_recordings, df, group_by_col='species_latin')
    if not table_1.empty:
        table_1.rename(columns={'species_latin': 'Bat Species (Normal Data)'}, inplace=True)
        
    table_2 = build_robust_ap_matrix(df_specialized_recordings, df, group_by_col='clean_specialized_group')
    if not table_2.empty:
        table_2.rename(columns={'clean_specialized_group': 'Specialized Call Variant'}, inplace=True)
        
    return table_1, table_2

# =====================================================================
# EXECUTION
# =====================================================================
dir = Path(os.getcwd()).resolve().parent / "data"
table_normal, table_specialized = generate_perfect_multilabel_tables(y_pred_proba_ood, str(dir / "ood_metadata.csv"))

print("\n" + "="*85)
print(" TABLE 1: PERFORMANCE ON NORMAL RECORDINGS (GROUPED BY SPECIES) ")
print("="*85)
print(table_normal.to_markdown(index=False))

print("\n" + "="*85)
print(" TABLE 2: PERFORMANCE ON SPECIALIZED RECORDINGS (WITH GLOBAL NEGATIVE CONTEXT) ")
print("="*85)
print(table_specialized.to_markdown(index=False))
print("\n Note: Scores marked with an asterisk (*) use global background samples to resolve single-class groups.")


 TABLE 1: PERFORMANCE ON NORMAL RECORDINGS (GROUPED BY SPECIES) 
| Bat Species (Normal Data)   | type_a AP   | type_b AP   | type_c AP   | type_d AP   | echo AP   |
|:----------------------------|:------------|:------------|:------------|:------------|:----------|
| Eptesicus serotinus         | 0.7067      | 0.8225      | 0.9926      | 0.9933*     | 0.9350    |
| Myotis dasycneme            | 0.5833      | 0.9672*     | 0.0462*     | 0.9888*     | 0.8875    |
| Myotis daubentonii          | 0.7942      | 0.9469*     | 0.8712      | 0.9898*     | 0.9924    |
| Nyctalus leisleri           | 0.0064*     | 0.9921*     | 0.9984*     | 0.9999*     | 0.0076*   |
| Nyctalus noctula            | 0.0808      | 0.1139      | 0.9189      | 0.8530      | 0.6163    |
| Pipistrellus nathusii       | 0.4816      | 0.5799      | 0.5897      | 0.9682      | 0.8425    |
| Pipistrellus pipistrellus   | 0.8571*     | 0.9627      | 0.6500      | 0.9869*     | 0.7008    |
| Pipistrellus pygmaeus       | 0.

In [21]:
import pandas as pd
import numpy as np
from sklearn.metrics import average_precision_score

def generate_species_centric_ap_tables(y_pred_proba, metadata_csv="ood_metadata.csv"):
    df = pd.read_csv(metadata_csv)
    class_cols = ['type_a', 'type_b', 'type_c', 'type_d', 'echo']
    
    # 1. Map model probabilities to the dataframe
    for idx, col in enumerate(class_cols):
        df[f'pred_{col}'] = y_pred_proba[:, idx]
        
    # 2. Define specialized keywords to split the data pools
    specialized_keywords = [
        'B.in_flight', 'C.i', 'C.d_stationary', 
        'C.d_flying', 'C.d_advertisement', 
        'D.thrills_nathusius', 'D.part_e_nathusius'
    ]
    
    # Isolate specialized recordings based on directory keywords
    df['is_specialized'] = df['subtype'].apply(lambda x: any(kw in str(x) for kw in specialized_keywords))
    
    # Split into two completely decoupled data environments
    df_normal_pool = df[~df['is_specialized']].copy()
    df_specialized_pool = df[df['is_specialized']].copy()
    
    # 3. Core AP Calculator (Purely local to the species group)
    def compute_species_ap(df_pool):
        rows = []
        for species, group_data in df_pool.groupby('species_latin'):
            # Ignore species with single isolated recordings to keep metrics stable
            if len(group_data) < 2:
                continue
                
            entry = {'Species': species}
            for col in class_cols:
                y_true = group_data[col].values
                y_pred = group_data[f'pred_{col}'].values
                
                # AP runs cleanly ONLY if the species contains both positive and negative examples
                if len(np.unique(y_true)) > 1:
                    ap_score = average_precision_score(y_true, y_pred)
                    entry[f'{col} AP'] = f"{ap_score:.4f}"
                else:
                    entry[f'{col} AP'] = "-" # Academically honest fallback for zero-variance classes
            rows.append(entry)
        return pd.DataFrame(rows)

    # 4. Generate the decoupled tables
    table_1 = compute_species_ap(df_normal_pool)
    table_2 = compute_species_ap(df_specialized_pool)
    
    return table_1, table_2

# =====================================================================
# EXECUTION
# =====================================================================
table_1_normal, table_2_specialized = generate_species_centric_ap_tables(y_pred_proba_ood, str(dir / "ood_metadata.csv"))

print("\n" + "="*85)
print(" TABLE 1: AP PER SPECIES (EVALUATED ONLY ON NORMAL RECORDINGS) ")
print("="*85)
print(table_1_normal.to_markdown(index=False) if not table_1_normal.empty else "No normal data available.")

print("\n" + "="*85)
print(" TABLE 2: AP PER SPECIES (EVALUATED ONLY ON SPECIALIZED RECORDINGS) ")
print("="*85)
print(table_2_specialized.to_markdown(index=False) if not table_2_specialized.empty else "No specialized data available.")


 TABLE 1: AP PER SPECIES (EVALUATED ONLY ON NORMAL RECORDINGS) 
| Species                   | type_a AP   | type_b AP   | type_c AP   | type_d AP   | echo AP   |
|:--------------------------|:------------|:------------|:------------|:------------|:----------|
| Eptesicus serotinus       | 0.7067      | 0.8225      | 0.9926      | -           | 0.9350    |
| Myotis dasycneme          | 0.5833      | -           | -           | -           | 0.8875    |
| Myotis daubentonii        | 0.7942      | -           | 0.8712      | -           | 0.9924    |
| Nyctalus noctula          | 0.0808      | 0.1139      | 0.9189      | 0.8530      | 0.6163    |
| Pipistrellus nathusii     | 0.4816      | 0.5799      | 0.5897      | 0.9682      | 0.8425    |
| Pipistrellus pipistrellus | -           | 0.9627      | 0.6500      | -           | 0.7008    |
| Pipistrellus pygmaeus     | -           | 1.0000      | -           | -           | -         |
| Plecotus auritus          | 0.1556      | 0.4631   

In [22]:
import pandas as pd
import numpy as np
from sklearn.metrics import average_precision_score

def generate_master_species_table(y_pred_proba, metadata_csv="ood_metadata.csv"):
    # 1. Load the complete consolidated metadata sheet
    df = pd.read_csv(metadata_csv)
    class_cols = ['type_a', 'type_b', 'type_c', 'type_d', 'echo']
    
    # 2. Map the model's predicted probabilities to the DataFrame
    for idx, col in enumerate(class_cols):
        df[f'pred_{col}'] = y_pred_proba[:, idx]
        
    # 3. Compute AP for each species using all available recordings
    master_rows = []
    for species, group_data in df.groupby('species_latin'):
        # Skip species with insufficient data (less than 2 recordings total)
        if len(group_data) < 2:
            continue
            
        entry = {'Bat Species': species}
        for col in class_cols:
            y_true = group_data[col].values
            y_pred = group_data[f'pred_{col}'].values
            
            # Compute AP only if the species has both positive and negative examples
            if len(np.unique(y_true)) > 1:
                ap_score = average_precision_score(y_true, y_pred)
                entry[f'{col.upper()} AP'] = f"{ap_score:.4f}"
            else:
                entry[f'{col.upper()} AP'] = "-" # Clean, academically honest indicator
                
        master_rows.append(entry)
        
    # 4. Convert to DataFrame and sort alphabetically by species name
    master_table = pd.DataFrame(master_rows)
    if not master_table.empty:
        master_table.sort_values(by='Bat Species', inplace=True)
        
    return master_table

# =====================================================================
# EXECUTION
# =====================================================================
# Pass your complete production out-of-distribution prediction matrix here:
master_evaluation_table = generate_master_species_table(y_pred_proba_ood, str(dir / "ood_metadata.csv"))

print("\n" + "="*90)
print(" MASTER EVALUATION TABLE: AVERAGE PRECISION (AP) PER SPECIES OVER ALL RECORDINGS ")
print("="*90)
print(master_evaluation_table.to_markdown(index=False))


 MASTER EVALUATION TABLE: AVERAGE PRECISION (AP) PER SPECIES OVER ALL RECORDINGS 
| Bat Species               | TYPE_A AP   | TYPE_B AP   | TYPE_C AP   | TYPE_D AP   | ECHO AP   |
|:--------------------------|:------------|:------------|:------------|:------------|:----------|
| Eptesicus serotinus       | 0.5479      | 0.6232      | 0.9407      | -           | 0.8168    |
| Myotis dasycneme          | 0.4693      | 0.5032      | 0.5548      | -           | 0.9589    |
| Myotis daubentonii        | 0.7174      | 0.5000      | 0.8637      | -           | 0.9359    |
| Nyctalus leisleri         | 1.0000      | -           | 0.8734      | -           | 0.1667    |
| Nyctalus noctula          | 0.0803      | 0.1072      | 0.9212      | 0.8489      | 0.6530    |
| Pipistrellus nathusii     | 0.3545      | 0.4974      | 0.4619      | 0.9770      | 0.9133    |
| Pipistrellus pipistrellus | -           | 0.6004      | 0.8707      | 0.1000      | 0.9181    |
| Pipistrellus pygmaeus     | -    

In [23]:
import pandas as pd

def convert_to_latex_table(master_table_df, label="tab:master_species_ap"):
    """
    Converts the master species performance DataFrame into publication-ready LaTeX code.
    Automatically applies italics to scientific names and uses booktabs formatting.
    """
    # 1. Create a deep copy to avoid mutating the original dataframe
    df_latex = master_table_df.copy()
    
    # 2. Automatically wrap the Latin names in standard LaTeX italics
    df_latex['Bat Species'] = df_latex['Bat Species'].apply(
        lambda x: f"\\textit{{{x}}}" if pd.notna(x) else x
    )
    
    # 3. Format the headers cleanly using LaTeX bold text styles
    header_mapping = {
        'Bat Species': '\\textbf{Bat Species}',
        'TYPE_A AP': '\\textbf{Type A AP}',
        'TYPE_B AP': '\\textbf{Type B AP}',
        'TYPE_C AP': '\\textbf{Type C AP}',
        'TYPE_D AP': '\\textbf{Type D AP}',
        'ECHO AP': '\\textbf{Echo AP}'
    }
    # Dynamic check in case columns are lowercase or uppercase
    header_mapping = {k: v for k, v in header_mapping.items() if k in df_latex.columns}
    df_latex.rename(columns=header_mapping, inplace=True)
    
    # 4. Generate the raw booktabs body
    # 'escape=False' is critical so LaTeX keywords like \textit are not escaped out
    latex_body = df_latex.to_latex(
        index=False,
        column_format='lccccc', # Left align species, center align the 5 metric columns
        escape=False
    )
    
    # 5. Build the wrapper table environment with captions and labels
    latex_wrapper = (
        "\\begin{table}[htbp]\n"
        "\\centering\n"
        "\\caption{Master Evaluation Table: Out-of-Distribution (OOD) performance measured "
        "via Average Precision (AP) per species across all recorded acoustic behaviors.}\n"
        f"\\label{{{label}}}\n"
        "\\small % Slightly reduces text size to fit cleanly on standard A4 layout\n"
        f"{latex_body}"
        "\\end{table}"
    )
    
    return latex_wrapper

# =====================================================================
# HOW TO RUN AND SAVE IT
# =====================================================================
# 1. Generate the string
latex_code_string = convert_to_latex_table(master_evaluation_table)

# 2. Print it straight to your console to copy/paste
print(latex_code_string)

# 3. Or save it directly to a .tex snippet file to import into your main document
with open("master_species_table.tex", "w") as f:
    f.write(latex_code_string)

\begin{table}[htbp]
\centering
\caption{Master Evaluation Table: Out-of-Distribution (OOD) performance measured via Average Precision (AP) per species across all recorded acoustic behaviors.}
\label{tab:master_species_ap}
\small % Slightly reduces text size to fit cleanly on standard A4 layout
\begin{tabular}{lccccc}
\toprule
\textbf{Bat Species} & \textbf{Type A AP} & \textbf{Type B AP} & \textbf{Type C AP} & \textbf{Type D AP} & \textbf{Echo AP} \\
\midrule
\textit{Eptesicus serotinus} & 0.5479 & 0.6232 & 0.9407 & - & 0.8168 \\
\textit{Myotis dasycneme} & 0.4693 & 0.5032 & 0.5548 & - & 0.9589 \\
\textit{Myotis daubentonii} & 0.7174 & 0.5000 & 0.8637 & - & 0.9359 \\
\textit{Nyctalus leisleri} & 1.0000 & - & 0.8734 & - & 0.1667 \\
\textit{Nyctalus noctula} & 0.0803 & 0.1072 & 0.9212 & 0.8489 & 0.6530 \\
\textit{Pipistrellus nathusii} & 0.3545 & 0.4974 & 0.4619 & 0.9770 & 0.9133 \\
\textit{Pipistrellus pipistrellus} & - & 0.6004 & 0.8707 & 0.1000 & 0.9181 \\
\textit{Pipistrellus pygmaeu

In [24]:
import pandas as pd
import numpy as np
from sklearn.metrics import average_precision_score

def generate_master_species_table_with_cmap(y_pred_proba, metadata_csv="ood_metadata.csv"):
    df = pd.read_csv(metadata_csv)
    class_cols = ['type_a', 'type_b', 'type_c', 'type_d', 'echo']
    
    for idx, col in enumerate(class_cols):
        df[f'pred_{col}'] = y_pred_proba[:, idx]
        
    master_rows = []
    for species, group_data in df.groupby('species_latin'):
        if len(group_data) < 2:
            continue
            
        entry = {'Bat Species': species}
        valid_aps = []  # Track numeric scores for this specific species row
        
        for col in class_cols:
            y_true = group_data[col].values
            y_pred = group_data[f'pred_{col}'].values
            
            if len(np.unique(y_true)) > 1:
                ap_score = average_precision_score(y_true, y_pred)
                entry[f'{col.upper()} AP'] = f"{ap_score:.4f}"
                valid_aps.append(ap_score)
            else:
                entry[f'{col.upper()} AP'] = "-"
        
        # Calculate the macro cmap over evaluated classes for this row
        if valid_aps:
            entry['cmap'] = f"{np.mean(valid_aps):.4f}"
        else:
            entry['cmap'] = "-"
            
        master_rows.append(entry)
        
    master_table = pd.DataFrame(master_rows)
    if not master_table.empty:
        master_table.sort_values(by='Bat Species', inplace=True)
        
    return master_table


def convert_to_latex_table(master_table_df, label="tab:master_species_ap"):
    df_latex = master_table_df.copy()
    
    # Apply scientific italics to Latin names
    df_latex['Bat Species'] = df_latex['Bat Species'].apply(
        lambda x: f"\\textit{{{x}}}" if pd.notna(x) else x
    )
    
    # Map headers to professional LaTeX formatting
    header_mapping = {
        'Bat Species': '\\textbf{Bat Species}',
        'TYPE_A AP': '\\textbf{Type A AP}',
        'TYPE_B AP': '\\textbf{Type B AP}',
        'TYPE_C AP': '\\textbf{Type C AP}',
        'TYPE_D AP': '\\textbf{Type D AP}',
        'ECHO AP': '\\textbf{Echo AP}',
        'cmap': '\\textbf{cmap}'
    }
    header_mapping = {k: v for k, v in header_mapping.items() if k in df_latex.columns}
    df_latex.rename(columns=header_mapping, inplace=True)
    
    # 'lcccccc' accommodates the species column + 5 class columns + 1 cmap column
    latex_body = df_latex.to_latex(
        index=False,
        column_format='lcccccc', 
        escape=False
    )
    
    latex_wrapper = (
        "\\begin{table}[htbp]\n"
        "\\centering\n"
        "\\caption{Master Evaluation Table: Out-of-Distribution (OOD) performance measured "
        "via Average Precision (AP) and class Mean Average Precision (cmap) per species.}\n"
        f"\\label{{{label}}}\n"
        "\\small\n"
        f"{latex_body}"
        "\\end{table}"
    )
    
    return latex_wrapper

# =====================================================================
# EXECUTION
# =====================================================================
# 1. Generate DataFrame with cmap column included
master_evaluation_table = generate_master_species_table_with_cmap(y_pred_proba_ood, str(dir / "ood_metadata.csv"))

# 2. Convert directly to the updated LaTeX block
latex_code_string = convert_to_latex_table(master_evaluation_table)
print(latex_code_string)

# 3. Or save it directly to a .tex snippet file to import into your main document
with open("master_species_table.tex", "w") as f:
    f.write(latex_code_string)

\begin{table}[htbp]
\centering
\caption{Master Evaluation Table: Out-of-Distribution (OOD) performance measured via Average Precision (AP) and class Mean Average Precision (cmap) per species.}
\label{tab:master_species_ap}
\small
\begin{tabular}{lcccccc}
\toprule
\textbf{Bat Species} & \textbf{Type A AP} & \textbf{Type B AP} & \textbf{Type C AP} & \textbf{Type D AP} & \textbf{Echo AP} & \textbf{cmap} \\
\midrule
\textit{Eptesicus serotinus} & 0.5479 & 0.6232 & 0.9407 & - & 0.8168 & 0.7321 \\
\textit{Myotis dasycneme} & 0.4693 & 0.5032 & 0.5548 & - & 0.9589 & 0.6216 \\
\textit{Myotis daubentonii} & 0.7174 & 0.5000 & 0.8637 & - & 0.9359 & 0.7542 \\
\textit{Nyctalus leisleri} & 1.0000 & - & 0.8734 & - & 0.1667 & 0.6800 \\
\textit{Nyctalus noctula} & 0.0803 & 0.1072 & 0.9212 & 0.8489 & 0.6530 & 0.5221 \\
\textit{Pipistrellus nathusii} & 0.3545 & 0.4974 & 0.4619 & 0.9770 & 0.9133 & 0.6408 \\
\textit{Pipistrellus pipistrellus} & - & 0.6004 & 0.8707 & 0.1000 & 0.9181 & 0.6223 \\
\textit{Pipis

In [25]:
import pandas as pd
import numpy as np
from sklearn.metrics import average_precision_score

def generate_master_species_table_with_totals(y_pred_proba, metadata_csv="ood_metadata.csv"):
    df = pd.read_csv(metadata_csv)
    class_cols = ['type_a', 'type_b', 'type_c', 'type_d', 'echo']
    
    # 1. Map model predictions
    for idx, col in enumerate(class_cols):
        df[f'pred_{col}'] = y_pred_proba[:, idx]
        
    # 2. Compute individual species rows
    species_rows = []
    for species, group_data in df.groupby('species_latin'):
        if len(group_data) < 2:
            continue
            
        entry = {'Bat Species': species}
        valid_aps = []
        
        for col in class_cols:
            y_true = group_data[col].values
            y_pred = group_data[f'pred_{col}'].values
            
            if len(np.unique(y_true)) > 1:
                ap_score = average_precision_score(y_true, y_pred)
                entry[f'{col.upper()} AP'] = f"{ap_score:.4f}"
                valid_aps.append(ap_score)
            else:
                entry[f'{col.upper()} AP'] = "-"
        
        if valid_aps:
            entry['cmap'] = f"{np.mean(valid_aps):.4f}"
        else:
            entry['cmap'] = "-"
            
        species_rows.append(entry)
        
    # Create and sort the species dataframe alphabetically
    master_table = pd.DataFrame(species_rows)
    if not master_table.empty:
        master_table.sort_values(by='Bat Species', inplace=True)
    
    # 3. Compute the GLOBAL summary row across ALL recordings
    global_entry = {'Bat Species': 'All Recordings'}
    global_aps = []
    
    for col in class_cols:
        y_true = df[col].values
        y_pred = df[f'pred_{col}'].values
        
        # Across the entire dataset, all classes will have 0s and 1s present
        global_ap = average_precision_score(y_true, y_pred)
        global_entry[f'{col.upper()} AP'] = f"{global_ap:.4f}"
        global_aps.append(global_ap)
        
    global_entry['cmap'] = f"{np.mean(global_aps):.4f}"
    
    # Append the summary row explicitly to the bottom
    master_table = pd.concat([master_table, pd.DataFrame([global_entry])], ignore_index=True)
    
    return master_table


def convert_to_latex_table(master_table_df, label="tab:master_species_ap"):
    df_latex = master_table_df.copy()
    
    # Scientific formatting: Italics for real species, Bold for the summary row
    df_latex['Bat Species'] = df_latex['Bat Species'].apply(
        lambda x: f"\\textbf{{{x}}}" if x == "All Recordings" else f"\\textit{{{x}}}"
    )
    
    # Map headers to clean academic styles
    header_mapping = {
        'Bat Species': '\\textbf{Bat Species}',
        'TYPE_A AP': '\\textbf{Type A AP}',
        'TYPE_B AP': '\\textbf{Type B AP}',
        'TYPE_C AP': '\\textbf{Type C AP}',
        'TYPE_D AP': '\\textbf{Type D AP}',
        'ECHO AP': '\\textbf{Echo AP}',
        'cmap': '\\textbf{cmAP}'
    }
    header_mapping = {k: v for k, v in header_mapping.items() if k in df_latex.columns}
    df_latex.rename(columns=header_mapping, inplace=True)
    
    # Generate the initial table body
    latex_body = df_latex.to_latex(
        index=False,
        column_format='lcccccc', 
        escape=False
    )
    
    # Inject a structural horizontal line directly above the summary row
    if "\\textbf{All Recordings}" in latex_body:
        latex_body = latex_body.replace(
            "\\textbf{All Recordings}", 
            "\\midrule\n\\textbf{All Recordings}"
        )
    
    latex_wrapper = (
        "\\begin{table}[htbp]\n"
        "\\centering\n"
        "\\caption{Master Evaluation Table: Out-of-Distribution (OOD) performance "
        "by species and global dataset configurations via Average Precision (AP) "
        "and class Mean Average Precision (cmAP).}\n"
        f"\\label{{{label}}}\n"
        "\\small\n"
        f"{latex_body}"
        "\\end{table}"
    )
    
    return latex_wrapper

# =====================================================================
# EXECUTION
# =====================================================================
# 1. Generate DataFrame with individual species and the bottom global totals row
master_evaluation_table = generate_master_species_table_with_totals(y_pred_proba_ood, str(dir / "ood_metadata.csv"))

# 2. Print Markdown check to terminal
print(master_evaluation_table.to_markdown(index=False))

# 3. Convert and output your final LaTeX code block
latex_code_string = convert_to_latex_table(master_evaluation_table)
print("\n" + "="*40 + " GENERATED LATEX CODE " + "="*40 + "\n")
print(latex_code_string)

# 3. Or save it directly to a .tex snippet file to import into your main document
with open("master_species_table.tex", "w") as f:
    f.write(latex_code_string)

| Bat Species               | TYPE_A AP   | TYPE_B AP   | TYPE_C AP   | TYPE_D AP   | ECHO AP   |   cmap |
|:--------------------------|:------------|:------------|:------------|:------------|:----------|-------:|
| Eptesicus serotinus       | 0.5479      | 0.6232      | 0.9407      | -           | 0.8168    | 0.7321 |
| Myotis dasycneme          | 0.4693      | 0.5032      | 0.5548      | -           | 0.9589    | 0.6216 |
| Myotis daubentonii        | 0.7174      | 0.5000      | 0.8637      | -           | 0.9359    | 0.7542 |
| Nyctalus leisleri         | 1.0000      | -           | 0.8734      | -           | 0.1667    | 0.68   |
| Nyctalus noctula          | 0.0803      | 0.1072      | 0.9212      | 0.8489      | 0.6530    | 0.5221 |
| Pipistrellus nathusii     | 0.3545      | 0.4974      | 0.4619      | 0.9770      | 0.9133    | 0.6408 |
| Pipistrellus pipistrellus | -           | 0.6004      | 0.8707      | 0.1000      | 0.9181    | 0.6223 |
| Pipistrellus pygmaeus     | -      